# 01 · Estímulos

**Objetivo.** Mostrar y validar las señales de prueba de la base común (paso P3 y P4 de la guía):

- bits de una **PRBS** (la misma que puede regenerar el hardware);
- **mapper Gray** PAM4 y QAM16 de potencia media 1;
- conformación **RRC** con el roll-off del escenario;
- el **receptor mínimo** (`rx.recover`) que se usa para medir, con constelación y ojo limpios.

Todo sale de `emulador/stimuli/` y `emulador/rx.py`, el mismo código que usan los tests (`tests/test_stimuli.py`).

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np

from emulador import plots
from emulador.common import rrc_taps
from emulador.config import Scenario
from emulador.metrics import evm_opt
from emulador.rx import matched_filter, recover
from emulador.stimuli import comm_signal, map_gray, prbs

plots.use_style()

## 1. Teoría

**PRBS.** Un LFSR de Fibonacci con polinomio primitivo $x^m + x^k + 1$ genera una secuencia de período $2^m - 1$ que cumple

$$
b[n] = b[n-m] \oplus b[n-k].
$$

En un período hay $2^{m-1}$ unos y $2^{m-1}-1$ ceros. Se usan PRBS-15 ($x^{15}+x^{14}+1$) y PRBS-31 ($x^{31}+x^{28}+1$), según ITU-T O.150. Los primeros $m$ bits de salida son la semilla, del MSB al LSB.

**Gray.** Etiquetas de símbolos vecinos que difieren en un solo bit: un error de decisión al vecino más cercano cuesta un bit. PAM4: 00 → −3, 01 → −1, 11 → +1, 10 → +3, normalizado por $\sqrt{5}$. QAM16: dos PAM4 Gray (I y Q), normalizado por $\sqrt{10}$.

**RRC.** El transmisor y el filtro adaptado son raíz de coseno realzado; su cascada es un coseno realzado, que cumple el criterio de Nyquist: sin ISI en los instantes $kT$. El espectro ocupa

$$
|f| \le \frac{(1+\rho)\,R_s}{2},
$$

o sea un ancho total de $(1+\rho)R_s$ (35,2 GHz con 32 GBd y $\rho = 0{,}1$).

**Piso de EVM.** El RRC truncado a *span* símbolos no es exactamente Nyquist. Con $\rho = 0{,}1$ la respuesta decae lento, y con span 32 el piso es 0,58 %. Por eso el span por defecto es 64 (decisión 15 de §1 de la guía).

## 2. Parámetros

In [ ]:
esc = Scenario()          # 32 GBd, 2 sps, roll-off 0,1
esc16 = Scenario(sps=16)  # solo para el ojo (resolución fina en el tiempo)
N_SYM = 8192
PRBS_ORDER = 15
SEED = 1

esc.to_dict()

## 3. PRBS

In [ ]:
period = 2**PRBS_ORDER - 1
b = prbs(PRBS_ORDER, 2 * period, seed=SEED)

print("primeros 32 bits:", "".join(map(str, b[:32])))
print("período         :", period, "->", np.array_equal(b[:period], b[period:]))
print("unos por período:", int(b[:period].sum()), "(2^14 =", 2**14, ")")

## 4. Mapper Gray

In [ ]:
labels = np.arange(16)
bits16 = ((labels[:, None] >> np.arange(3, -1, -1)) & 1).ravel()
pts = map_gray(bits16, "qam16")

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(pts.real, pts.imag, "o", ms=8)
for lab, p in zip(labels, pts):
    ax.annotate(f"{lab:04b}", (p.real, p.imag), textcoords="offset points", xytext=(0, 9),
                ha="center", fontsize=10)
ax.set_aspect("equal")
ax.set_xlim(-1.3, 1.3)
ax.set_ylim(-1.3, 1.3)
ax.set_xlabel("I")
ax.set_ylabel("Q")
ax.set_title("QAM16 Gray (b0 b1 -> I, b2 b3 -> Q)")
plots.save_fig(fig, "01_qam16_gray")

print("potencia media QAM16:", np.mean(np.abs(pts) ** 2))
print("potencia media PAM4 :", np.mean(np.abs(map_gray([0, 0, 0, 1, 1, 1, 1, 0], 'pam4')) ** 2))

## 5. Señales

In [ ]:
sig_qam = comm_signal(esc, mod="qam16", n_symbols=N_SYM, prbs_order=PRBS_ORDER, seed=SEED)
sig_pam = comm_signal(esc, mod="pam4", n_symbols=N_SYM, prbs_order=PRBS_ORDER, seed=SEED)
sig_pam16 = comm_signal(esc16, mod="pam4", n_symbols=N_SYM, prbs_order=PRBS_ORDER, seed=SEED)

sig_qam.meta()

## 6. Espectro

Las líneas verticales marcan $\pm(1+\rho)R_s/2$. Se mide la fracción de potencia fuera de esa banda.

In [ ]:
from scipy.signal import welch

edge = (1 + esc.rolloff) * esc.Rs / 2
f, p = welch(sig_qam.samples, fs=esc.fs, nperseg=4096, return_onesided=False)
out_db = 10 * np.log10(p[np.abs(f) > 1.02 * edge].sum() / p.sum())
print(f"ancho teórico (1+rho)Rs = {2 * edge / 1e9:.1f} GHz")
print(f"potencia fuera de banda  = {out_db:.1f} dB")

fig, ax = plt.subplots()
plots.spectrum(ax, sig_qam.samples, esc.fs, nperseg=4096, label="QAM16")
for s in (-1, 1):
    ax.axvline(s * edge / 1e9, color="C1", ls="--", lw=1)
ax.set_ylim(-200, None)
ax.set_title(f"Espectro RRC, rho = {esc.rolloff}")
plots.save_fig(fig, "01_espectro_rrc")

## 7. Receptor mínimo: constelaciones back-to-back

In [ ]:
r_qam = recover(sig_qam.samples, sig_qam)
r_pam = recover(sig_pam.samples, sig_pam)

fig, axs = plt.subplots(1, 2, figsize=(10, 4.8))
plots.constellation(axs[0], r_qam.symbols, ref=r_qam.ref, title=f"QAM16 · EVM {100 * r_qam.evm:.2f} %")
plots.constellation(axs[1], r_pam.symbols, ref=r_pam.ref, title=f"PAM4 · EVM {100 * r_pam.evm:.2f} %")
plots.save_fig(fig, "01_constelaciones_b2b")

print("QAM16: fase", r_qam.phase, "retardo", r_qam.delay, f"EVM {100 * r_qam.evm:.3f} %")
print("PAM4 : fase", r_pam.phase, "retardo", r_pam.delay, f"EVM {100 * r_pam.evm:.3f} %")

## 8. Ojo PAM4

A la salida del **transmisor** el pulso es un RRC y el ojo no está limpio (el RRC solo no cumple Nyquist). Después del **filtro adaptado** el pulso total es un coseno realzado y el ojo queda abierto en los instantes $kT$. Se usa sps = 16 para tener resolución en el tiempo.

In [ ]:
sps = esc16.sps
y16 = matched_filter(sig_pam16.samples, esc16, sig_pam16.span)
y16 /= np.sqrt(np.mean(np.abs(y16[::sps]) ** 2))
skip = 2 * sig_pam16.span * sps          # descarta transitorios
off = skip + sps // 2                    # ojo centrado: instantes kT en t = 0,5 y 1,5

fig, axs = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
x_tx = sig_pam16.samples / np.sqrt(np.mean(np.abs(sig_pam16.samples[::sps]) ** 2))
plots.eye(axs[0], x_tx, sps, n_traces=250, offset=off, title="Salida del TX (RRC)")
plots.eye(axs[1], y16, sps, n_traces=250, offset=off, title="Tras el filtro adaptado (RC)")
for ax in axs:
    for lv in np.array([-3, -1, 1, 3]) / np.sqrt(5):
        ax.axhline(lv, color="C1", lw=0.8, ls=":")
plots.save_fig(fig, "01_ojo_pam4", dpi=110)

## 9. Piso de EVM contra el span del RRC (decisión 15)

In [ ]:
print(" span   EVM b2b")
floors = {}
for span in (32, 48, 64, 96):
    s = comm_signal(esc, n_symbols=N_SYM, span=span)
    floors[span] = recover(s.samples, s).evm
    print(f"{span:5d}   {100 * floors[span]:.3f} %")

## 10. Validación

Los mismos criterios que `tests/test_stimuli.py`.

In [ ]:
assert np.array_equal(b[:period], b[period:])
assert int(b[:period].sum()) == 2**14
assert np.isclose(np.mean(np.abs(pts) ** 2), 1.0, atol=1e-12)
assert np.isclose(np.mean(np.abs(sig_qam.samples) ** 2), 1.0, atol=1e-12)
assert (r_qam.phase, r_qam.delay) == (0, 0)
assert 100 * r_qam.evm < 0.5 and 100 * r_pam.evm < 0.5
assert out_db < -40
assert floors[32] > 0.005 > floors[64]
print("OK")

## 11. Conclusiones

- La PRBS-15 tiene período $2^{15}-1$ y $2^{14}$ unos por período; los bits son reproducibles en el RTL a partir de la semilla.
- El mapper Gray da constelaciones de potencia 1 con vecinos a un bit de distancia.
- El espectro RRC ocupa $(1+\rho)R_s$ = 35,2 GHz; la potencia fuera de banda está por debajo de −40 dB.
- Con span 64 el piso de EVM back-to-back es ~0,12 %, bien por debajo del criterio de 0,5 %. Con span 32 sería 0,58 % (decisión 15).
- El receptor mínimo recupera fase de muestreo y retardo; es la base para medir ISI y CD en los notebooks 02 a 04.

Figuras en `docs/golden_model/figuras/01_*.png`.